# Lines

**Geometry type:** `line`

Independent two-point segments, such as synaptic contacts (pre- and
post-synaptic sites), vessel endpoints or distance annotations.

1. Generate synthetic synapse-like segments
2. Write them to a `.zv` store
3. Open the store and inspect its metadata
4. Read every segment and its attributes
5. Read by object id and by bounding box
6. Validate

In [1]:
import os
import tempfile

import numpy as np
import zarr_vectors as zv
from zarr_vectors.types.lines import read_lines, write_lines
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
STORE = os.path.join(_tmpdir, "synapses.zv")
print("Store path:", STORE)

Store path: /tmp/zv_examples_4_lggbyd/synapses.zv


## 1 · Generate synthetic contact sites

5,000 segments with midpoints in a 400 µm cube; each runs between a pre- and a
post-synaptic site up to 2.6 µm either side of the midpoint.

In [2]:
rng = np.random.default_rng(0)
N = 5_000

midpoints = rng.uniform(50, 450, (N, 3)).astype(np.float32)   # µm
offsets = rng.uniform(-1.5, 1.5, (N, 3)).astype(np.float32)

pre_pos  = midpoints - offsets
post_pos = midpoints + offsets

# write_lines takes endpoints shaped (N, 2, D): [i, 0] = pre, [i, 1] = post.
endpoints = np.stack([pre_pos, post_pos], axis=1).astype(np.float32)

weights = rng.uniform(0, 1, (N, 2)).astype(np.float32)        # per endpoint
synapse_type = rng.integers(0, 4, N).astype(np.int32)         # per line

seg_lengths = np.linalg.norm(endpoints[:, 1] - endpoints[:, 0], axis=1)
print(f"endpoints    : {endpoints.shape}  (N, 2, D)")
print(f"weights      : {weights.shape}  (one per endpoint)")
print(f"synapse_type : {synapse_type.shape}  (one per line)")
print(f"length       : {seg_lengths.min():.2f} – {seg_lengths.max():.2f} µm")

endpoints    : (5000, 2, 3)  (N, 2, D)
weights      : (5000, 2)  (one per endpoint)
synapse_type : (5000,)  (one per line)
length       : 0.18 – 4.95 µm


## 2 · Write the store

Each line is one object. A line whose endpoints fall in different chunks is
stored with one endpoint in each chunk and a cross-chunk link between them;
`cross_chunk_count` counts those.

In [3]:
summary = write_lines(
    STORE,
    endpoints,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    vertex_attributes={"weight": weights},
    object_attributes={"synapse_type": synapse_type},
)
print(summary)

{'line_count': 5000, 'chunk_count': 125, 'cross_chunk_count': 197}


## 3 · Open and inspect

In [4]:
ds = zv.open(STORE)
lv0 = ds.level(0)
print(ds)
print(f"kinds        : {ds.kinds}")
print(f"chunk shape  : {lv0.scale}")
print(f"vertex_count : {lv0.vertex_count:,}  (= {N:,} lines × 2 endpoints)")
print(f"objects      : {lv0.objects.count:,}")
print(f"vertex attrs : {lv0.attribute_names('vertex')}")
print(f"object attrs : {lv0.attribute_names('object')}")

Dataset('file:///tmp/zv_examples_4_lggbyd/synapses.zv', line, levels=[0])
kinds        : ('line',)
chunk shape  : (100.0, 100.0, 100.0)
vertex_count : 10,000  (= 5,000 lines × 2 endpoints)
objects      : 5,000
vertex attrs : ('weight',)
object attrs : ('synapse_type',)


## 4 · Read all segments

`read_lines` returns `endpoints` as `(M, 2, D)` in object-id order, and each
vertex attribute flattened to one value per endpoint, `(2M,)`. Object
attributes are read per level with `read_object_attributes`.

In [5]:
result = read_lines(STORE)
print(f"line_count : {result['line_count']:,}")
print(f"endpoints  : {result['endpoints'].shape}")

lengths = np.linalg.norm(result["endpoints"][:, 1] - result["endpoints"][:, 0], axis=1)
print(f"length     : mean {lengths.mean():.2f} µm, std {lengths.std():.2f} µm")

w = result["vertex_attributes"]["weight"].reshape(-1, 2)
print(f"weight     : {w.shape}, matches input: {np.allclose(w, weights)}")

from zarr_vectors.building import get_resolution_level, open_store, read_object_attributes

level0 = get_resolution_level(open_store(STORE), 0)
stype = read_object_attributes(level0, "synapse_type")
print(f"synapse_type counts: {np.bincount(stype).tolist()}")

line_count : 5,000
endpoints  : (5000, 2, 3)
length     : mean 2.86 µm, std 0.84 µm
weight     : (5000, 2), matches input: True
synapse_type counts: [1248, 1248, 1240, 1264]


## 5 · Read by object id and by bounding box

A bounding-box read returns every line with at least one endpoint inside the
box, whole: both endpoints, even when one lies outside.

In [6]:
by_id = read_lines(STORE, object_ids=[0, 42])
print(f"Lines 0 and 42: {by_id['line_count']}, match input: "
      f"{np.allclose(by_id['endpoints'], endpoints[[0, 42]])}")

lo = np.array([200.0, 200.0, 200.0])
hi = np.array([300.0, 300.0, 300.0])
bbox_result = read_lines(STORE, bbox=(lo, hi))

ep = bbox_result["endpoints"]                       # (M, 2, 3)
in_box = ((ep >= lo) & (ep <= hi)).all(axis=2)      # (M, 2)
print(f"\nLines returned for the 100 µm box : {bbox_result['line_count']}")
print(f"  with ≥1 endpoint inside          : {in_box.any(axis=1).sum()}")
print(f"  with both endpoints inside       : {in_box.all(axis=1).sum()}")

truth = ((endpoints >= lo) & (endpoints <= hi)).all(axis=2).any(axis=1).sum()
print(f"Input lines with ≥1 endpoint inside: {truth}")

Lines 0 and 42: 2, match input: True

Lines returned for the 100 µm box : 79
  with ≥1 endpoint inside          : 79
  with both endpoints inside       : 74
Input lines with ≥1 endpoint inside: 79


## 6 · Validate

In [7]:
print(validate(STORE, level=3).summary())

Level 3 validation: PASS
  26 passed, 0 warnings, 0 errors


## Summary

Each line is one object of two vertices; lines may cross chunk boundaries.
Use `polyline` for paths of more than two vertices.

| Step | API |
|------|-----|
| Write | `write_lines(path, endpoints, chunk_shape, bin_shape, vertex_attributes=..., object_attributes=...)` |
| Open | `zarr_vectors.open(path)` → `Dataset` |
| Read all | `read_lines(path)` → `endpoints (M, 2, D)`, `vertex_attributes`, `line_count` |
| Read by id | `read_lines(path, object_ids=[...])` |
| Bbox query | `read_lines(path, bbox=(lo, hi))` (lines with ≥1 endpoint inside) |
| Object attributes | `zarr_vectors.building.read_object_attributes(level_group, name)` |